# Compare old trainer data vs Lightning DataModule

This notebook compares data preparation between the legacy pipeline ([src/trainer.py](../src/trainer.py)) and the Lightning pipeline ([src/lightning_data.py](../src/lightning_data.py)).

In [6]:
import os
import sys
import yaml
import numpy as np
import torch

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import data_util
from lightning_data import CaloINNDataModule

torch.set_default_dtype(torch.float32)
np.random.seed(0)
torch.manual_seed(0)

print('repo_root =', repo_root)

repo_root = /global/cfs/cdirs/m3443/usr/pmtuan/caloinn_lightning


In [7]:
old_cfg_path = os.path.join(repo_root, 'params', 'pions.yaml')
new_cfg_path = os.path.join(repo_root, 'params', 'lightning_pion.yaml')

with open(old_cfg_path, 'r') as f:
    old_cfg = yaml.safe_load(f)
with open(new_cfg_path, 'r') as f:
    new_cfg = yaml.safe_load(f)

new_data = new_cfg['data']
new_ds_kwargs = new_data['dataset_kwargs']
new_model = new_cfg['model']

mapping = {
    'data_path': (old_cfg.get('data_path'), new_data.get('data_path')),
    'val_data_path': (old_cfg.get('val_data_path'), new_data.get('val_data_path')),
    'batch_size': (old_cfg.get('batch_size'), new_data.get('batch_size')),
    'val_frac': (old_cfg.get('val_frac'), new_data.get('val_frac')),
    'xml_path': (old_cfg.get('xml_path'), new_ds_kwargs.get('xml_path')),
    'xml_ptype': (old_cfg.get('xml_ptype'), new_ds_kwargs.get('xml_ptype')),
    'single_energy': (old_cfg.get('single_energy'), new_ds_kwargs.get('single_energy')),
    'eps': (old_cfg.get('eps'), new_ds_kwargs.get('eps')),
    'u0up_cut': (old_cfg.get('u0up_cut'), new_ds_kwargs.get('u0up_cut')),
    'u0low_cut': (old_cfg.get('u0low_cut'), new_ds_kwargs.get('u0low_cut')),
    'pt_rew': (old_cfg.get('pt_rew'), new_ds_kwargs.get('pt_rew')),
    'dep_cut': (old_cfg.get('dep_cut'), new_ds_kwargs.get('dep_cut')),
    'width_noise': (old_cfg.get('width_noise'), new_model.get('width_noise')),
    'lr': (old_cfg.get('lr'), new_cfg['optimizer']['init_args'].get('lr')),
    'max_lr': (old_cfg.get('max_lr'), new_cfg['lr_scheduler']['init_args'].get('max_lr')),
    'weight_decay': (old_cfg.get('weight_decay'), new_cfg['optimizer']['init_args'].get('weight_decay')),
    'betas': (old_cfg.get('betas'), new_cfg['optimizer']['init_args'].get('betas')),
    'n_epochs': (old_cfg.get('n_epochs'), new_cfg['trainer'].get('max_epochs')),
}

print('Config compatibility check (old vs lightning):')
mismatches = []
for k, (v_old, v_new) in mapping.items():
    ok = v_old == v_new
    print(f'- {k}: {v_old} | {v_new} | match={ok}')
    if not ok:
        mismatches.append(k)

print('\nMismatches:', mismatches if mismatches else 'None')

Config compatibility check (old vs lightning):
- data_path: /pscratch/sd/p/pmtuan/calochallenge/ds1-pions/dataset_1_pions_2.hdf5 | /pscratch/sd/p/pmtuan/calochallenge/ds1-pions/dataset_1_pions_2.hdf5 | match=True
- val_data_path: /pscratch/sd/p/pmtuan/calochallenge/ds1-pions/dataset_1_pions_2.hdf5 | /pscratch/sd/p/pmtuan/calochallenge/ds1-pions/dataset_1_pions_2.hdf5 | match=True
- batch_size: 512 | 512 | match=True
- val_frac: 0.01 | 0.01 | match=True
- xml_path: /global/cfs/cdirs/m3443/usr/pmtuan/caloinn_lightning/binning_dataset_1_pions.xml | /global/cfs/cdirs/m3443/usr/pmtuan/caloinn_lightning/binning_dataset_1_pions.xml | match=True
- xml_ptype: pion | pion | match=True
- single_energy: None | None | match=True
- eps: 1e-10 | 1e-10 | match=True
- u0up_cut: 3.5 | 3.5 | match=True
- u0low_cut: 0.0 | 0.0 | match=True
- pt_rew: 1.0 | 1.0 | match=True
- dep_cut: 600 | 600 | match=True
- width_noise: 5e-06 | 5e-06 | match=True
- lr: 1e-05 | 1e-05 | match=True
- max_lr: 0.0001 | 0.0001 |

## 1) Same raw index range -> same prepared tensors?

This directly validates whether the new dataset class applies the same `load_data` + `preprocess` logic as the legacy pipeline when given the same raw events.

In [8]:
seed = 2026
torch.manual_seed(seed)
np.random.seed(seed)

# Legacy reference loader
old_train_loader, _, _ = data_util.get_loaders(
    old_cfg.get('data_path'),
    old_cfg.get('xml_path'),
    old_cfg.get('xml_ptype'),
    old_cfg.get('val_frac'),
    old_cfg.get('batch_size'),
    old_cfg.get('eps', 1.0e-10),
    device='cpu',
    shuffle=old_cfg.get('shuffle', False),
    width_noise=old_cfg.get('width_noise', 1e-7),
    energy=old_cfg.get('single_energy', None),
    u0up_cut=old_cfg.get('u0up_cut', 7.0),
    u0low_cut=old_cfg.get('u0low_cut', 0.0),
    rew=old_cfg.get('pt_rew', 1.0),
    dep_cut=old_cfg.get('dep_cut', 1.0e10),
)

# Lightning DataModule now reuses the same legacy loading path in setup('fit')
torch.manual_seed(seed)
np.random.seed(seed)
dm = CaloINNDataModule(**new_data)
dm.setup('fit')
new_train_loader = dm.train_dataloader()

# Reseed before each iterator creation to compare exact same random permutation/noise draws
torch.manual_seed(seed)
np.random.seed(seed)
old_x, old_c = next(iter(old_train_loader))

torch.manual_seed(seed)
np.random.seed(seed)
new_x, new_c = next(iter(new_train_loader))

print('old shape x,c:', tuple(old_x.shape), tuple(old_c.shape))
print('new shape x,c:', tuple(new_x.shape), tuple(new_c.shape))

max_abs_x = float(torch.max(torch.abs(old_x - new_x)).item())
max_abs_c = float(torch.max(torch.abs(old_c - new_c)).item())
print('max_abs_diff x:', max_abs_x)
print('max_abs_diff c:', max_abs_c)
print('allclose x (atol=1e-7):', bool(torch.allclose(old_x, new_x, atol=1e-7, rtol=0.0)))
print('allclose c (atol=1e-12):', bool(torch.allclose(old_c, new_c, atol=1e-12, rtol=0.0)))

print('num_train_samples old/new:', len(old_train_loader.data), dm.num_train_samples)

old shape x,c: (512, 540) (512, 1)
new shape x,c: (512, 540) (512, 1)
max_abs_diff x: 0.0
max_abs_diff c: 0.0
allclose x (atol=1e-7): True
allclose c (atol=1e-12): True
num_train_samples old/new: 119516 119516


## 2) Compare first training batch (old trainer vs Lightning datamodule)

This checks what enters each training loop path. Note: old `MyDataLoader` adds noise in the loader, while Lightning adds noise in `training_step`.

In [8]:
np.random.seed(0)

torch.manual_seed(0)



xml_path = old_cfg.get('xml_path')

xml_abs = xml_path

if xml_abs.startswith('./'):

    xml_abs = os.path.join(repo_root, xml_abs[2:])

xml_abs = os.path.abspath(xml_abs)



# IMPORTANT: disable legacy loader noise to compare pure loader outputs

old_train_loader, old_val_loader, _ = data_util.get_loaders(

    old_cfg.get('data_path'),

    xml_abs,

    old_cfg.get('xml_ptype'),

    old_cfg.get('val_frac'),

    old_cfg.get('batch_size'),

    old_cfg.get('eps'),

    device='cpu',

    width_noise=0.0,

    energy=old_cfg.get('single_energy', None),

    u0up_cut=old_cfg.get('u0up_cut', 7.0),

    u0low_cut=old_cfg.get('u0low_cut', 0.0),

    rew=old_cfg.get('pt_rew', 1.0),

    dep_cut=old_cfg.get('dep_cut', 1e10),

    shuffle=False,

)



old_x, old_c = next(iter(old_train_loader))



new_ds_kwargs_local = dict(new_ds_kwargs)

new_ds_kwargs_local['xml_path'] = xml_abs



dm = CaloINNDataModule(

    data_path=new_data['data_path'],

    val_data_path=new_data['val_data_path'],

    batch_size=new_data['batch_size'],

    cond_key=new_data.get('cond_key', 'incident_energies'),

    sample_key=new_data.get('sample_key', 'showers'),

    val_frac=new_data.get('val_frac', 0.01),

    shuffle=False,

    eval_dataset=new_data.get('eval_dataset', '1-pions'),

    num_workers=0,

    predict_batch_size=new_data.get('predict_batch_size', 1000),

    dataset_kwargs=new_ds_kwargs_local,

)

dm.setup('fit')

new_x, new_c = next(iter(dm.train_dataloader()))



print('old first batch x,c:', tuple(old_x.shape), tuple(old_c.shape))

print('new first batch x,c:', tuple(new_x.shape), tuple(new_c.shape))



if old_x.shape == new_x.shape and old_c.shape == new_c.shape:

    max_abs_c = torch.max(torch.abs(old_c - new_c)).item()

    max_abs_x = torch.max(torch.abs(old_x - new_x)).item()

    allclose_c = torch.allclose(old_c, new_c, rtol=0.0, atol=1e-12)

    allclose_x = torch.allclose(old_x, new_x, rtol=0.0, atol=1e-7)



    print('cond max_abs_diff:', max_abs_c)

    print('x max_abs_diff   :', max_abs_x)

    print('allclose cond    :', allclose_c)

    print('allclose x       :', allclose_x)

else:

    print('Batch shapes differ, likely due to split semantics differences.')


old first batch x,c: (512, 540) (512, 1)
new first batch x,c: (512, 540) (512, 1)
cond max_abs_diff: 0.0
x max_abs_diff   : 0.0
allclose cond    : True
allclose x       : True


## 3) Compare train/val split semantics (full loaders, no noise)

This section compares how samples are assigned to train vs val in the old pipeline and in the Lightning datamodule.

Both sides use `shuffle=False` and `width_noise=0.0` to isolate split logic only.

In [9]:
from collections import Counter

def _collect_all(loader):
    xs, cs = [], []
    for xb, cb in loader:
        xs.append(xb.detach().cpu())
        cs.append(cb.detach().cpu())
    if not xs:
        return torch.empty(0, 0), torch.empty(0, 0)
    return torch.cat(xs, dim=0), torch.cat(cs, dim=0)

def _row_counter(x_t, c_t, decimals=8):
    arr = torch.cat([x_t, c_t], dim=1).numpy()
    arr = np.round(arr, decimals=decimals)
    return Counter(row.tobytes() for row in arr)

# Old pipeline split (after preprocess), no noise
old_train_loader, old_val_loader, _ = data_util.get_loaders(
    old_cfg.get('data_path'),
    xml_abs,
    old_cfg.get('xml_ptype'),
    old_cfg.get('val_frac'),
    old_cfg.get('batch_size'),
    old_cfg.get('eps'),
    device='cpu',
    width_noise=0.0,
    energy=old_cfg.get('single_energy', None),
    u0up_cut=old_cfg.get('u0up_cut', 7.0),
    u0low_cut=old_cfg.get('u0low_cut', 0.0),
    rew=old_cfg.get('pt_rew', 1.0),
    dep_cut=old_cfg.get('dep_cut', 1e10),
    shuffle=False,
)

old_train_x_all = torch.clone(old_train_loader.data).cpu()
old_train_c_all = torch.clone(old_train_loader.cond).cpu()
old_val_x_all = torch.clone(old_val_loader.data).cpu()
old_val_c_all = torch.clone(old_val_loader.cond).cpu()

# Lightning split (before preprocess), no internal shuffle
dm_split = CaloINNDataModule(
    data_path=new_data['data_path'],
    val_data_path=new_data['val_data_path'],
    batch_size=new_data['batch_size'],
    cond_key=new_data.get('cond_key', 'incident_energies'),
    sample_key=new_data.get('sample_key', 'showers'),
    val_frac=new_data.get('val_frac', 0.01),
    shuffle=False,
    eval_dataset=new_data.get('eval_dataset', '1-pions'),
    num_workers=0,
    predict_batch_size=new_data.get('predict_batch_size', 1000),
    dataset_kwargs=new_ds_kwargs_local,
)
dm_split.setup('fit')

new_train_x_all, new_train_c_all = _collect_all(dm_split.train_dataloader())
new_val_x_all, new_val_c_all = _collect_all(dm_split.val_dataloader())

# Compare counts
print('Counts:')
print(f"- old train: {old_train_x_all.shape[0]}")
print(f"- old val  : {old_val_x_all.shape[0]}")
print(f"- new train: {new_train_x_all.shape[0]}")
print(f"- new val  : {new_val_x_all.shape[0]}")

# Compare assignment overlap as multisets of rows
old_train_ctr = _row_counter(old_train_x_all, old_train_c_all)
old_val_ctr = _row_counter(old_val_x_all, old_val_c_all)
new_train_ctr = _row_counter(new_train_x_all, new_train_c_all)
new_val_ctr = _row_counter(new_val_x_all, new_val_c_all)

train_overlap = sum((old_train_ctr & new_train_ctr).values())
val_overlap = sum((old_val_ctr & new_val_ctr).values())

old_train_only = sum((old_train_ctr - new_train_ctr).values())
new_train_only = sum((new_train_ctr - old_train_ctr).values())
old_val_only = sum((old_val_ctr - new_val_ctr).values())
new_val_only = sum((new_val_ctr - old_val_ctr).values())

# Union consistency check
old_union = old_train_ctr + old_val_ctr
new_union = new_train_ctr + new_val_ctr
union_equal = old_union == new_union

print('\nAssignment overlap:')
print(f"- train overlap: {train_overlap} / {old_train_x_all.shape[0]}")
print(f"- val overlap  : {val_overlap} / {old_val_x_all.shape[0]}")
print(f"- old_train only vs new_train: {old_train_only} vs {new_train_only}")
print(f"- old_val only   vs new_val  : {old_val_only} vs {new_val_only}")

print('\nUnion consistency (old train+val == new train+val):', union_equal)

Counts:
- old train: 119516
- old val  : 1207
- new train: 119515
- new val  : 1208

Assignment overlap:
- train overlap: 119515 / 119516
- val overlap  : 1207 / 1207
- old_train only vs new_train: 1 vs 0
- old_val only   vs new_val  : 0 vs 1

Union consistency (old train+val == new train+val): True


## 4) Summary helper
Run this cell after the previous ones for a compact verdict.

In [7]:
print('If section 1 shows allclose=True for x and c, preprocessing logic matches for same raw events.')
print('If section 2 differs in x but not c, that is expected from noise location differences and/or split semantics.')

If section 1 shows allclose=True for x and c, preprocessing logic matches for same raw events.
If section 2 differs in x but not c, that is expected from noise location differences and/or split semantics.
